In [2]:
import torch
from torch import nn
import math

In [9]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super(MultiheadAttention, self).__init__()

        assert embed_dim % num_heads == 0, "embed_dim phải chia hết cho num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout = dropout

        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, attn_mask=None):
        batch_size, seq_length, embed_dim = query.size()

        Q = self.q_proj(query)
        K = self.k_proj(key)
        V = self.v_proj(value)

        # .view(): Ép [B, T, d_model] thành [B, T, num_heads, d_k]
        # .transpose(1, 2): Đổi chỗ chiều T và num_heads -> [B, num_heads, T, d_k]
        # Thao tác lật này cực kỳ quan trọng để lát nữa nhân ma trận không bị nhầm lẫn giữa các head.
        Q = Q.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        # Lật 2 chiều cuối của K: [B, num_heads, d_k, T_k]
        # Phép nhân: [B, num_heads, T_q, d_k] x [B, num_heads, d_k, T_k] 
        # Kết quả (scores): [B, num_heads, T_q, T_k]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attn_mask is not None:
            scores += attn_mask

        attn_weights = torch.softmax(scores, dim=-1)
        # attn_weights = torch.dropout(attn_weights, p=self.dropout)

        # [B, num_heads, T_q, T_k] x [B, num_heads, T_k, d_k] 
        # Kết quả : [B, num_heads, T_q, d_k]
        attn_output = torch.matmul(attn_weights, V)

        attn_output = attn_output.transpose(1, 2).contiguous()
        
        # Ép phẳng lại thành 3 chiều: [B, T_q, d_model]
        attn_output = attn_output.view(batch_size, seq_length, self.embed_dim)

        output = self.out_proj(attn_output)

        return output, attn_weights

In [10]:
mha = MultiheadAttention(embed_dim=512, num_heads=8)
query = torch.rand(32, 10, 512)  # (batch_size, seq_length, embed_dim)
key = torch.rand(32, 10, 512) 
value = torch.rand(32, 10, 512)
output, attn_weights = mha(query, key, value)
print("Output shape:", output.shape)  # (32, 10, 512)

Output shape: torch.Size([32, 10, 512])


In [11]:
B = 16         # Batch size
T = 256         # Sequence length
embed_dim = 512 
num_heads = 8

mha = MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads)
X = torch.rand(B, T, embed_dim)

bool_mask = torch.tril(torch.ones(T, T))

# 2. Tạo một ma trận chứa toàn số 0.0 cùng kích thước
# Sử dụng torch.zeros để có kiểu dữ liệu là float
additive_mask = torch.zeros(T, T)

# 3. Dùng masked_fill_ để lấp đầy phần bị che (nơi bool_mask == 0) bằng -inf
additive_mask = additive_mask.masked_fill(bool_mask == 0, float('-inf'))

print("Ma trận Additive Mask (Sẵn sàng để truyền vào mạng):")
print(additive_mask)
print("-" * 50)

# 4. TRUYỀN VÀO MÔ HÌNH
# Lưu ý quan trọng: Shape của scores là [B, num_heads, T, T]
# Ta cần reshape mask thành [1, 1, T, T] để Pytorch tự động phát sóng (broadcast) cho tất cả các Head và Batch.
additive_mask = additive_mask.unsqueeze(0).unsqueeze(0)

output, weights = mha(query=X, key=X, value=X, attn_mask=additive_mask)

print("Shape của Output:", output.shape)
print("Trọng số Attention (Câu 1, Head 1):")
print(torch.round(weights[0, 0] * 1000) / 1000)

Ma trận Additive Mask (Sẵn sàng để truyền vào mạng):
tensor([[0., -inf, -inf,  ..., -inf, -inf, -inf],
        [0., 0., -inf,  ..., -inf, -inf, -inf],
        [0., 0., 0.,  ..., -inf, -inf, -inf],
        ...,
        [0., 0., 0.,  ..., 0., -inf, -inf],
        [0., 0., 0.,  ..., 0., 0., -inf],
        [0., 0., 0.,  ..., 0., 0., 0.]])
--------------------------------------------------
Shape của Output: torch.Size([16, 256, 512])
Trọng số Attention (Câu 1, Head 1):
tensor([[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.4790, 0.5210, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3270, 0.3380, 0.3350,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0000, 0.0000],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0000],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040]],
       grad_fn=<DivBackward0>)
